In [1]:
import os
import sys
import numpy as np
import torch
import random

In [6]:
random.random()

0.9755003976628528

In [78]:
x = torch.randn(1000)
n = 0.5*torch.randn(1000)

In [79]:
initial_snr = x.pow(2).sum() / n.pow(2).sum()
initial_snr

tensor(3.8537)

In [80]:
desired_snr = 3
desired_snr = 10**(0.1*desired_snr)

In [81]:
n = n * (initial_snr / desired_snr).sqrt()

In [82]:
x.pow(2).sum() / n.pow(2).sum()

tensor(1.9953)

In [4]:
root = '/home/ovistetom/Documents/Databases_Local/MIXTURES/standard'

In [14]:
for subset_name in ['trn', 'val', 'tst']:
    subset_path = os.path.join(root, subset_name)
    for sample_name in os.listdir(subset_path):
        sample_path = os.path.join(subset_path, sample_name)
        metadata_path = os.path.join(sample_path, 'metadata.txt')
        with open(metadata_path, 'r') as f:
            lines = f.readlines()
        break
    break




In [15]:
sample_echo = lines[-1].split(',')[1].strip().startswith('T')

In [16]:
sample_echo

False

In [17]:
lines

['speak,P254\n',
 'distr,1748\n',
 'noise,STRAFFIC\n',
 'distrSNR,+INF\n',
 'noiseSNR,-19\n',
 'echo,FALSE\n']

In [20]:
float('-INF')

-inf

In [ ]:
import numpy as np
import room_acoustics_utils as utils
import pyroomacoustics as pra
import random
import matplotlib.pyplot as plt

In [ ]:
room_dim = np.array([0.5, 0.5, 0.5])
head_pos = 0.5*room_dim

In [ ]:
head_yaw = 0.25*np.pi
rot_yaw = np.array([[np.cos(head_yaw), -np.sin(head_yaw), 0.0], [np.sin(head_yaw), np.cos(head_yaw), 0.0], [0.0, 0.0, 1.0]])

In [ ]:
rot_yaw @ head_pos

In [ ]:
# def random_mouth_position(head_pos, head_yaw):
#     hpx, hpy, hpz = head_pos
#     rdx = random.uniform(-0.01, 0.01)
#     rdy = random.uniform(0.11, 0.15)
#     rdz = random.uniform(-0.04, -0.02)
#     x = hpx + rdx*np.cos(head_yaw) - rdy*np.sin(head_yaw)
#     y = hpy + rdx*np.sin(head_yaw) + rdy*np.cos(head_yaw)
#     z = hpz + rdz
#     return np.array([x, y, z])

def random_mouth_position(head_pos, head_yaw=0, head_pitch=0, head_roll=0):
    # Define initial mouth position relative to head center.
    rdx = random.uniform(0.11, 0.15)
    rdy = random.uniform(-0.01, 0.01)
    rdz = random.uniform(-0.04, -0.02)
    mouth_pos = np.array([rdx, rdy, rdz])
    # Define rotation matrix around vertical axis (yaw).
    rot_yaw = np.array([[np.cos(head_yaw), -np.sin(head_yaw), 0.0], [np.sin(head_yaw), np.cos(head_yaw), 0.0], [0.0, 0.0, 1.0]])
    # Define rotation matrix around lateral axis (pitch).
    rot_pitch = np.array([[np.cos(head_pitch), 0.0, np.sin(head_pitch)], [0.0, 1.0, 0.0], [-np.sin(head_pitch), 0.0, np.cos(head_pitch)]])
    # Define rotation matrix around horizontal axis (roll).
    rot_roll = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(head_roll), -np.sin(head_roll)], [0.0, np.sin(head_roll), np.cos(head_roll)]])
    # Rotate mouth position.
    mouth_pos = rot_roll @ rot_pitch @ rot_yaw @ mouth_pos
    # Translate to head position.
    mouth_pos = mouth_pos + head_pos
    return mouth_pos


In [ ]:
mouth_pos = random_mouth_position(head_pos, head_yaw=0.8, head_pitch=0, head_roll=0.8)

In [ ]:
room = pra.ShoeBox(room_dim)
room.add_source(head_pos)
room.add_microphone(mouth_pos)
room.plot();

In [ ]:
# VIEW FROM ABOVE (Z PLANE)
room = pra.ShoeBox(room_dim[:2])
room.add_source(head_pos[:2])
room.add_microphone(mouth_pos[:2])
room.plot();

In [ ]:
# VIEW FROM FRONT (X PLANE)
room = pra.ShoeBox(room_dim[1:])
room.add_source(head_pos[1:])
room.add_microphone(mouth_pos[1:])
room.plot();

In [ ]:
# VIEW FROM RIGHT SIDE (Y PLANE)
room = pra.ShoeBox(room_dim[0::2])
room.add_source(head_pos[0::2])
room.add_microphone(mouth_pos[0::2])
room.plot();

In [ ]:
def random_mouth_position(head_pos, head_yaw=0, head_pitch=0, head_roll=0):
    # Define initial mouth position relative to head center.
    rdx = random.uniform(0.11, 0.15)
    rdy = random.uniform(-0.01, 0.01)
    rdz = random.uniform(-0.04, -0.02)
    mouth_pos = np.array([rdx, rdy, rdz])
    # Define rotation matrix around vertical axis (yaw).
    rot_yaw = np.array([[np.cos(head_yaw), -np.sin(head_yaw), 0.0], [np.sin(head_yaw), np.cos(head_yaw), 0.0], [0.0, 0.0, 1.0]])
    # Define rotation matrix around lateral axis (pitch).
    rot_pitch = np.array([[np.cos(head_pitch), 0.0, np.sin(head_pitch)], [0.0, 1.0, 0.0], [-np.sin(head_pitch), 0.0, np.cos(head_pitch)]])
    # Define rotation matrix around horizontal axis (roll).
    rot_roll = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(head_roll), -np.sin(head_roll)], [0.0, np.sin(head_roll), np.cos(head_roll)]])
    # Rotate mouth position.
    mouth_pos = np.einsum('ik, i -> k', rot_roll @ rot_pitch @ rot_yaw, mouth_pos)
    # Translate to head position.
    mouth_pos = mouth_pos + head_pos
    return mouth_pos

def define_mics_position(head_pos, head_yaw=0, head_pitch=0, head_roll=0):
    rdy = random.uniform(0.08, 0.09)
    # Define initial position of mics relative to head center.
    mic_l_dn = np.array([0.0, + rdy, - 0.01])
    mic_l_up = np.array([0.0, + rdy, + 0.01])    
    mic_r_dn = np.array([0.0, - rdy, - 0.01])
    mic_r_up = np.array([0.0, - rdy, + 0.01])
    mics_pos = np.array([mic_l_dn, mic_l_up, mic_r_dn, mic_r_up])
    # Define rotation matrix around vertical axis (yaw).
    rot_yaw = np.array([[np.cos(head_yaw), -np.sin(head_yaw), 0.0], [np.sin(head_yaw), np.cos(head_yaw), 0.0], [0.0, 0.0, 1.0]])
    # Define rotation matrix around lateral axis (pitch).
    rot_pitch = np.array([[np.cos(head_pitch), 0.0, np.sin(head_pitch)], [0.0, 1.0, 0.0], [-np.sin(head_pitch), 0.0, np.cos(head_pitch)]])
    # Define rotation matrix around horizontal axis (roll).
    rot_roll = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(head_roll), -np.sin(head_roll)], [0.0, np.sin(head_roll), np.cos(head_roll)]])
    # Rotate position of mics.
    mics_pos = np.einsum('ik, mi -> mk', rot_roll @ rot_pitch @ rot_yaw, mics_pos)
    # Translate position of mics.
    mics_pos = mics_pos + np.repeat(head_pos[None, ...], repeats=4, axis=0)
    return mics_pos


In [ ]:
head_yaw, head_pitch, head_roll = 3.14, 1.6, 1.6
mouth_pos = random_mouth_position(head_pos, head_yaw=head_yaw, head_pitch=head_pitch, head_roll=head_roll)
mics_pos = define_mics_position(head_pos, head_yaw=head_yaw, head_pitch=head_pitch, head_roll=head_roll)

In [ ]:
# Create room and add necessary sources.
room = pra.ShoeBox(room_dim)
room.add_source(head_pos)
room.add_source(mouth_pos)
room.add_microphone(mics_pos.T)
room.add_microphone(mics_pos.T)
room.plot();

In [ ]:
# VIEW FROM ABOVE (Z PLANE)
room = pra.ShoeBox(room_dim[:2])
room.add_source(head_pos[:2])
room.add_source(mouth_pos[:2])
room.add_microphone(mics_pos.T[:2])
room.add_microphone(mics_pos.T[:2])
room.plot();

In [ ]:
# VIEW FROM FRONT (X PLANE)
room = pra.ShoeBox(room_dim[1:])
room.add_source(head_pos[1:])
room.add_source(mouth_pos[1:])
room.add_microphone(mics_pos.T[1:])
room.add_microphone(mics_pos.T[1:])
room.plot();

In [ ]:
# VIEW FROM RIGHT SIDE (Y PLANE)
room = pra.ShoeBox(room_dim[0::2])
room.add_source(head_pos[0::2])
room.add_source(mouth_pos[0::2])
room.add_microphone(mics_pos.T[0::2])
room.add_microphone(mics_pos.T[0::2])
room.plot();